# SAR Ship Detection — End-to-End Pipeline (v3)

This notebook rebuilds the ship-detection project as one linear, reusable pipeline. It replaces `Ship_detection_2_0 (3).ipynb`, which grew into 113 cells (many empty, duplicated, or broken mid-edit) and had gotten hard to follow or rerun.



**Structure:**
1. Setup & global configuration
2. SSDD dataset preparation (COCO → YOLO)
3. Baseline model E0 (YOLO11s)
4. E1 — SensorMix resolution-robustness experiment
5. Sentinel-1 real-world pipeline (reusable functions)
6. Multi-scene detection + AIS validation (pre-filter)
7. False-alarm filtering (post-processing) + re-validation
8. Results, conclusions, next steps

> Run cells top to bottom. Part 2-4 (SSDD training) only need to run once per model version — if `E0`/`E1` weights already exist on Drive, you can skip straight to Part 5.


## Part 0 — Setup

In [ ]:
# Install packages not preinstalled in Colab.
# (rasterio/asf_search/global-land-mask are the ones actually missing; the rest ship with Colab.)
!pip install -q ultralytics gdown lxml asf_search rasterio global-land-mask

import torch
import ultralytics

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Ultralytics:", ultralytics.__version__)

from google.colab import drive
drive.mount('/content/drive')


PyTorch: 2.11.0+cpu | CUDA available: False
Ultralytics: 8.4.129


ValueError: mount failed

In [ ]:
from pathlib import Path

# ============================================================================
# GLOBAL CONFIGURATION
#
# Every path / threshold used below is defined here, once. If you need to
# point at a different dataset, AOI, date range, or tune a filter, change it
# here — nothing later in the notebook hardcodes its own copy.
# ============================================================================

class CFG:
    # ---- Drive / local paths -------------------------------------------------
    DRIVE_ROOT   = Path("/content/drive/MyDrive")
    PROJECT_ROOT = DRIVE_ROOT / "SAR_Ship_Project"
    MODEL_DIR    = PROJECT_ROOT / "models"          # trained weights live here
    RESULTS_DIR  = PROJECT_ROOT / "results_v3"       # everything this notebook produces

    RAR_FILE     = DRIVE_ROOT / "Official_SSDD.rar"
    LOCAL_DATA   = Path("/content/Official_SSDD")    # extracted SSDD
    YOLO_ROOT    = Path("/content/SSDD_YOLO")        # YOLO-format dataset
    E1_ROOT      = Path("/content/SSDD_SensorMix")   # E1 augmented dataset

    S1_ROOT      = Path("/content/Sentinel1_scenes") # all downloaded/processed S1 scenes

    # ---- SSDD split -----------------------------------------------------------
    VAL_FRACTION = 0.15   # fraction of the official "train" split held out as val
    SPLIT_SEED   = 42

    # ---- Training ---------------------------------------------------------
    BASE_WEIGHTS = "yolo11s.pt"
    EPOCHS       = 50
    PATIENCE     = 10
    IMGSZ        = 640
    BATCH        = 16

    # ---- Area of interest: Singapore Strait shipping corridor ----------------
    LON_MIN, LON_MAX = 103.55, 104.15
    LAT_MIN, LAT_MAX = 1.05, 1.30

    AOI_WKT = (
        f"POLYGON(({LON_MIN} {LAT_MIN}, {LON_MAX} {LAT_MIN}, "
        f"{LON_MAX} {LAT_MAX}, {LON_MIN} {LAT_MAX}, {LON_MIN} {LAT_MIN}))"
    )
    AOI_GEOJSON = {
        "type": "Polygon",
        "coordinates": [[
            [LON_MIN, LAT_MIN], [LON_MAX, LAT_MIN],
            [LON_MAX, LAT_MAX], [LON_MIN, LAT_MAX], [LON_MIN, LAT_MIN],
        ]],
    }

    # ---- Sentinel-1 search / selection ----------------------------------------
    SEARCH_START      = "2024-06-01T00:00:00Z"
    SEARCH_END        = "2026-08-20T23:59:59Z"
    BEAM_MODE         = "IW"
    POLARIZATION      = "VV+VH"
    N_SCENES_TO_RUN   = 12         # how many scenes the multi-scene pipeline actually processes
    PREFER_SAME_GEOMETRY = True    # prefer one flight direction/path so scenes are comparable

    # ---- Sentinel-1 processing --------------------------------------------
    CROP_BUFFER_PX   = 100         # padding (px) around AOI when cropping the raw scene
    DB_MIN, DB_MAX   = -30.0, 5.0  # sigma0 dB clip range used to build the 8-bit model image
    PIXEL_SPACING_M  = 10.0        # Sentinel-1 IW GRD ground pixel spacing (approx, for size filters)

    # ---- Tiled inference --------------------------------------------------
    TILE_SIZE      = 640
    TILE_STRIDE    = 512           # 20% overlap
    CONF_THRESHOLD = 0.25
    NMS_IOU        = 0.50

    # ---- AIS (Global Fishing Watch) ----------------------------------------
    GFW_REPORT_URL     = "https://gateway.api.globalfishingwatch.org/v3/4wings/report"
    GFW_AIS_DATASET    = "public-global-presence:latest"
    AIS_MATCH_RADII_M  = [500, 750, 1000, 1500]
    AIS_MAIN_RADIUS_M  = 1000
    AIS_TIME_TOLERANCE_HOURS = 1.5   # how far an AIS record's time bin may sit from the SAR
                                      # acquisition time and still count as "supporting" it

    # ---- False-alarm filters (post-processing) ------------------------------
    LAND_BUFFER_M           = 150     # detections within this distance of the coast are kept
                                       # even if the land mask flags the exact point (jetties,
                                       # anchorages, coastal noise in the mask itself)
    CLUTTER_CLUSTER_EPS_M   = 120      # DBSCAN radius for grouping repeated detections across scenes
    CLUTTER_MIN_SCENE_FRAC  = 0.6      # a location seen in >= 60% of processed scenes is "static"
    CLUTTER_MIN_SCENES      = 3        # ...but only evaluated once we have at least this many scenes

    MIN_VESSEL_LENGTH_M = 15.0
    MAX_VESSEL_LENGTH_M = 400.0
    MIN_ASPECT_RATIO    = 1.3          # length / width; near-square boxes are usually not ships
    MAX_ASPECT_RATIO    = 12.0

    for p in [MODEL_DIR, RESULTS_DIR, S1_ROOT]:
        p.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print("AOI:", CFG.AOI_WKT.strip())


## Part 1 — SSDD dataset preparation

Extract the official SSDD (SAR Ship Detection Dataset) release, inspect its COCO annotations, split it into train/val/test, and convert it to YOLO format.

In [ ]:
import json

# ---- Extract the RAR (idempotent: skip if already extracted) --------------
already_extracted = CFG.LOCAL_DATA.exists() and any(CFG.LOCAL_DATA.iterdir())

if not already_extracted:
    CFG.LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    !apt-get update -qq && apt-get install -y -qq unrar
    !unrar x -o+ "{CFG.RAR_FILE}" "{CFG.LOCAL_DATA}/"
else:
    print("SSDD already extracted, skipping unrar.")

print("\nTop-level contents of", CFG.LOCAL_DATA)
for item in CFG.LOCAL_DATA.iterdir():
    print(" -", item.name)

# The official release nests everything under one folder; find it.
DATASET_ROOT = CFG.LOCAL_DATA
candidates = list(CFG.LOCAL_DATA.rglob("BBox_SSDD"))
if candidates:
    DATASET_ROOT = candidates[0].parent

BBOX_ROOT    = DATASET_ROOT / "BBox_SSDD" / "coco_style"
TRAIN_IMAGES = BBOX_ROOT / "images" / "train"
TEST_IMAGES  = BBOX_ROOT / "images" / "test"
TRAIN_JSON   = BBOX_ROOT / "annotations" / "train.json"
TEST_JSON    = BBOX_ROOT / "annotations" / "test.json"

print("\nBBOX_ROOT:", BBOX_ROOT)
assert TRAIN_JSON.exists() and TEST_JSON.exists(), "Could not locate SSDD COCO annotations — check the extracted folder layout above."

with open(TRAIN_JSON) as f:
    train_data = json.load(f)
with open(TEST_JSON) as f:
    test_data = json.load(f)

print("\nSSDD COCO summary")
print(f"  Train images      : {len(train_data['images'])}")
print(f"  Train annotations : {len(train_data['annotations'])}  (ships; can exceed image count)")
print(f"  Test images       : {len(test_data['images'])}")
print(f"  Test annotations  : {len(test_data['annotations'])}")

In [ ]:
import random

def group_annotations(coco_data):
    """image_id -> list of its annotation dicts."""
    grouped = {}
    for ann in coco_data["annotations"]:
        grouped.setdefault(ann["image_id"], []).append(ann)
    return grouped

train_annotations_all = group_annotations(train_data)
test_annotations       = group_annotations(test_data)

# Split the official "train" images into train/val (test stays untouched).
random.seed(CFG.SPLIT_SEED)
all_train_images = train_data["images"].copy()
random.shuffle(all_train_images)

n_val = int(len(all_train_images) * CFG.VAL_FRACTION)
val_images   = all_train_images[:n_val]
train_images = all_train_images[n_val:]
test_images  = test_data["images"]

train_annotations = train_annotations_all
val_annotations   = train_annotations_all  # same lookup table; filtered by image id per split

print(f"Train images: {len(train_images)}")
print(f"Val images  : {len(val_images)}")
print(f"Test images : {len(test_images)}")

In [ ]:
import shutil

for split in ["train", "val", "test"]:
    (CFG.YOLO_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (CFG.YOLO_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)


def convert_to_yolo(image_records, annotation_lookup, source_image_folder, split_name):
    """Copy images + write YOLO-format .txt labels (single class: 'ship') for one split."""
    dst_img_dir = CFG.YOLO_ROOT / "images" / split_name
    dst_lbl_dir = CFG.YOLO_ROOT / "labels" / split_name

    for img in image_records:
        w, h = img["width"], img["height"]
        src = source_image_folder / img["file_name"]
        if not src.exists():
            continue
        shutil.copy(src, dst_img_dir / img["file_name"])

        lines = []
        for ann in annotation_lookup.get(img["id"], []):
            x, y, bw, bh = ann["bbox"]  # COCO: top-left x, y, width, height
            cx, cy = (x + bw / 2) / w, (y + bh / 2) / h
            nw, nh = bw / w, bh / h
            lines.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")

        label_path = dst_lbl_dir / (Path(img["file_name"]).stem + ".txt")
        label_path.write_text("\n".join(lines))

    return len(image_records)


n_train = convert_to_yolo(train_images, train_annotations, TRAIN_IMAGES, "train")
n_val   = convert_to_yolo(val_images,   val_annotations,   TRAIN_IMAGES, "val")
n_test  = convert_to_yolo(test_images,  test_annotations,  TEST_IMAGES,  "test")

print("Converted to YOLO format:")
for split in ["train", "val", "test"]:
    n_imgs = len(list((CFG.YOLO_ROOT / "images" / split).glob("*.jpg")))
    print(f"  {split:5s}: {n_imgs} images")

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ---- YOLO dataset yaml -----------------------------------------------------
YAML_PATH = CFG.YOLO_ROOT / "ssdd.yaml"
YAML_PATH.write_text(
    f"path: {CFG.YOLO_ROOT}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"test: images/test\n"
    f"names:\n  0: ship\n"
)
print("Wrote", YAML_PATH)
print(YAML_PATH.read_text())

# ---- Sanity check: plot a few images with their converted boxes -----------
sample_files = random.sample(list((CFG.YOLO_ROOT / "images" / "train").glob("*.jpg")), 4)

fig, axes = plt.subplots(1, len(sample_files), figsize=(5 * len(sample_files), 5))
for ax, img_path in zip(axes, sample_files):
    img = Image.open(img_path)
    w, h = img.size
    ax.imshow(img, cmap="gray")

    label_path = CFG.YOLO_ROOT / "labels" / "train" / (img_path.stem + ".txt")
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        _, cx, cy, nw, nh = map(float, line.split())
        x1, y1 = (cx - nw / 2) * w, (cy - nh / 2) * h
        ax.add_patch(patches.Rectangle((x1, y1), nw * w, nh * h, fill=False, edgecolor="lime", linewidth=1.5))

    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Part 2 — Baseline model (E0)

Train YOLO11s on SSDD as-is. This is the reference model everything else is compared against.

In [ ]:
from ultralytics import YOLO

E0_WEIGHTS = CFG.MODEL_DIR / "E0_YOLO11s_SSDD_Baseline.pt"

if E0_WEIGHTS.exists():
    print("E0 weights already exist on Drive, loading:", E0_WEIGHTS)
    model_e0 = YOLO(str(E0_WEIGHTS))
else:
    model_e0 = YOLO(CFG.BASE_WEIGHTS)
    results_e0 = model_e0.train(
        data=str(YAML_PATH),
        epochs=CFG.EPOCHS,
        patience=CFG.PATIENCE,
        imgsz=CFG.IMGSZ,
        batch=CFG.BATCH,
        device=0,
        project="/content/SAR_Ship_Experiments",
        name="E0_YOLO11s_Baseline",
    )
    best_pt = Path(model_e0.trainer.best)
    shutil.copy(best_pt, E0_WEIGHTS)
    model_e0 = YOLO(str(E0_WEIGHTS))
    print("Saved E0 weights to:", E0_WEIGHTS)

In [ ]:
# ---- Evaluate E0 on the held-out SSDD test split --------------------------
metrics_e0 = model_e0.val(data=str(YAML_PATH), split="test", device=0)

print("E0 baseline — SSDD test metrics")
print(f"  mAP50    : {metrics_e0.box.map50:.4f}")
print(f"  mAP50-95 : {metrics_e0.box.map:.4f}")
print(f"  Precision: {metrics_e0.box.mp:.4f}")
print(f"  Recall   : {metrics_e0.box.mr:.4f}")

## Part 3 — E1: SensorMix resolution-robustness experiment

SSDD images are all a fairly uniform resolution/quality. Real-world Sentinel-1 scenes vary a lot more. E1 trains on a 50/50 mix of original SSDD images and "SensorMix" versions of the same images (resolution degradation + speckle + dynamic-range + blur variation, geometry untouched) so the model has seen more of that variation before it ever sees Sentinel-1 data.

In [ ]:
import cv2
import numpy as np

# ============================================================================
# SensorMix augmentation — simulates cross-sensor / cross-resolution variation
# without touching bounding-box geometry, so YOLO labels stay valid as-is.
# ============================================================================

def degrade_resolution(image, rng, forced_level=None):
    """Blur -> downsample -> upsample back, at one of three severities."""
    h, w = image.shape
    levels = {
        "mild":     ((0.60, 0.80), (0.6, 1.0)),
        "moderate": ((0.35, 0.60), (1.0, 1.6)),
        "strong":   ((0.18, 0.35), (1.5, 2.4)),
    }
    if forced_level is None:
        p = rng.random()
        forced_level = "mild" if p < 0.30 else ("moderate" if p < 0.75 else "strong")

    (scale_lo, scale_hi), (psf_lo, psf_hi) = levels[forced_level]
    scale = rng.uniform(scale_lo, scale_hi)
    preblur_sigma = rng.uniform(psf_lo, psf_hi)

    blurred = cv2.GaussianBlur(image, (0, 0), sigmaX=preblur_sigma)
    small_w, small_h = max(8, int(w * scale)), max(8, int(h * scale))
    small = cv2.resize(blurred, (small_w, small_h), interpolation=cv2.INTER_AREA)
    degraded = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    return degraded, scale, forced_level, preblur_sigma


def add_speckle(image, rng):
    """Multiplicative gamma-distributed noise (SAR speckle)."""
    looks = rng.uniform(2.5, 6.0)
    noise = rng.gamma(shape=looks, scale=1.0 / looks, size=image.shape).astype(np.float32)
    output = np.clip(image.astype(np.float32) * noise, 0, 255)
    return output.astype(np.uint8), looks


def change_dynamic_range(image, rng):
    """Gamma + gain shift, simulating intensity/contrast differences across sensors."""
    img = image.astype(np.float32) / 255.0
    gamma, gain = rng.uniform(0.75, 1.30), rng.uniform(0.85, 1.15)
    output = np.clip(gain * (img ** gamma), 0, 1)
    return (output * 255).astype(np.uint8), gamma, gain


def add_blur(image, rng):
    sigma = rng.uniform(0.3, 1.0)
    return cv2.GaussianBlur(image, (0, 0), sigmaX=sigma), sigma


def sensor_mix(image, seed, return_info=False):
    """One SensorMix image: mandatory resolution degradation + optional speckle
    (60%) / dynamic-range change (60%) / extra blur (20%)."""
    rng = np.random.default_rng(seed)
    output = image.copy()
    applied = []

    output, scale, level, psf = degrade_resolution(output, rng)
    applied.append(f"resolution_{level}(scale={scale:.2f}, psf_sigma={psf:.2f})")

    if rng.random() < 0.60:
        output, looks = add_speckle(output, rng)
        applied.append(f"speckle(L={looks:.2f})")

    if rng.random() < 0.60:
        output, gamma, gain = change_dynamic_range(output, rng)
        applied.append(f"dynamic(gamma={gamma:.2f}, gain={gain:.2f})")

    if rng.random() < 0.20:
        output, sigma = add_blur(output, rng)
        applied.append(f"extra_blur(sigma={sigma:.2f})")

    return (output, applied) if return_info else output


# ---- Quick visual check on one random training image ----------------------
sample_path = random.choice(list((CFG.YOLO_ROOT / "images" / "train").glob("*.jpg")))
sample_img = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)
mixed, applied = sensor_mix(sample_img, seed=999, return_info=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
axes[0].imshow(sample_img, cmap="gray"); axes[0].set_title("Original"); axes[0].axis("off")
axes[1].imshow(mixed, cmap="gray"); axes[1].set_title("SensorMix\n" + "\n".join(applied), fontsize=8); axes[1].axis("off")
plt.suptitle(sample_path.name)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# Build the E1 dataset: 50% original + 50% SensorMix training images.
# Val/test stay untouched (SSDD original) so E0 and E1 remain comparable.
# ============================================================================

if CFG.E1_ROOT.exists():
    shutil.rmtree(CFG.E1_ROOT)
for split in ["train", "val", "test"]:
    (CFG.E1_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (CFG.E1_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

train_imgs = sorted((CFG.YOLO_ROOT / "images" / "train").glob("*.jpg"))
random.seed(CFG.SPLIT_SEED)
shuffled = train_imgs.copy()
random.shuffle(shuffled)
half = len(shuffled) // 2
keep_original, make_sensormix = shuffled[:half], shuffled[half:]

n_original = n_sensormix = 0
for img_path in keep_original:
    label_path = CFG.YOLO_ROOT / "labels" / "train" / (img_path.stem + ".txt")
    if not label_path.exists():
        continue
    shutil.copy2(img_path, CFG.E1_ROOT / "images" / "train" / img_path.name)
    shutil.copy2(label_path, CFG.E1_ROOT / "labels" / "train" / label_path.name)
    n_original += 1

for i, img_path in enumerate(make_sensormix):
    label_path = CFG.YOLO_ROOT / "labels" / "train" / (img_path.stem + ".txt")
    if not label_path.exists():
        continue
    image = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    augmented = sensor_mix(image, seed=10_000 + i)
    out_name = f"{img_path.stem}_sensormix"
    cv2.imwrite(str(CFG.E1_ROOT / "images" / "train" / f"{out_name}.jpg"), augmented)
    # Geometry is unchanged by SensorMix, so the original label file is reused as-is.
    shutil.copy2(label_path, CFG.E1_ROOT / "labels" / "train" / f"{out_name}.txt")
    n_sensormix += 1

# Val/test: copy through unchanged.
for split in ["val", "test"]:
    for img_path in (CFG.YOLO_ROOT / "images" / split).glob("*.jpg"):
        shutil.copy2(img_path, CFG.E1_ROOT / "images" / split / img_path.name)
    for label_path in (CFG.YOLO_ROOT / "labels" / split).glob("*.txt"):
        shutil.copy2(label_path, CFG.E1_ROOT / "labels" / split / label_path.name)

E1_YAML = CFG.E1_ROOT / "ssdd_e1.yaml"
E1_YAML.write_text(
    f"path: {CFG.E1_ROOT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: ship\n"
)

print(f"E1 train set: {n_original} original + {n_sensormix} SensorMix = {n_original + n_sensormix} images")
print("Wrote", E1_YAML)

In [ ]:
E1_WEIGHTS = CFG.MODEL_DIR / "E1_YOLO11s_SSDD_SensorMix.pt"

if E1_WEIGHTS.exists():
    print("E1 weights already exist on Drive, loading:", E1_WEIGHTS)
    model_e1 = YOLO(str(E1_WEIGHTS))
else:
    model_e1 = YOLO(CFG.BASE_WEIGHTS)
    results_e1 = model_e1.train(
        data=str(E1_YAML),
        epochs=CFG.EPOCHS,
        patience=CFG.PATIENCE,
        imgsz=CFG.IMGSZ,
        batch=CFG.BATCH,
        device=0,
        seed=CFG.SPLIT_SEED,
        project="/content/SAR_Ship_Experiments",
        name="E1_YOLO11s_SensorMix",
    )
    best_pt = Path(model_e1.trainer.best)
    shutil.copy(best_pt, E1_WEIGHTS)
    model_e1 = YOLO(str(E1_WEIGHTS))
    print("Saved E1 weights to:", E1_WEIGHTS)

In [ ]:
import pandas as pd

# E1 is evaluated on its own yaml (same val/test images, just a different path root).
metrics_e1 = model_e1.val(data=str(E1_YAML), split="test", device=0)

comparison_ssdd = pd.DataFrame([
    {"Model": "E0 Baseline", "mAP50": metrics_e0.box.map50, "mAP50-95": metrics_e0.box.map,
     "Precision": metrics_e0.box.mp, "Recall": metrics_e0.box.mr},
    {"Model": "E1 SensorMix", "mAP50": metrics_e1.box.map50, "mAP50-95": metrics_e1.box.map,
     "Precision": metrics_e1.box.mp, "Recall": metrics_e1.box.mr},
])
comparison_ssdd.to_csv(CFG.RESULTS_DIR / "E0_E1_SSDD_test_comparison.csv", index=False)
comparison_ssdd

## Part 4 — Sentinel-1 real-world pipeline (reusable functions)

Everything below is written as a function that takes a scene/product as an argument. v2 had this logic inline, hardcoded to one manually-picked scene (`results[3]`) and one hardcoded date — so it could only ever be run once, by hand, and never rerun on a new image. Here it's built once and called in a loop in Part 6.

Pipeline per scene: download → extract `.SAFE` → find VV (and VH) rasters → radiometric calibration (DN → σ0 → dB) → crop to the Singapore Strait AOI → 8-bit model image → tiled YOLO inference (E0 & E1) → global NMS → pixel → lon/lat geolocation → AIS presence match.

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
import rasterio
from scipy.interpolate import griddata

# ============================================================================
# XML helpers (Sentinel-1 SAFE annotation/calibration files use namespaced tags)
# ============================================================================

def local_tag(element):
    return element.tag.split("}")[-1]

def child_text(element, name):
    for child in element:
        if local_tag(child) == name:
            return child.text
    return None


def read_geolocation_grid(annotation_xml):
    """Returns an [N, 4] array of [pixel, line, longitude, latitude] tie points."""
    points = []
    for elem in ET.parse(annotation_xml).getroot().iter():
        if local_tag(elem) != "geolocationGridPoint":
            continue
        pixel, line = child_text(elem, "pixel"), child_text(elem, "line")
        lat, lon = child_text(elem, "latitude"), child_text(elem, "longitude")
        if None in (pixel, line, lat, lon):
            continue
        points.append([float(pixel), float(line), float(lon), float(lat)])
    points = np.array(points, dtype=np.float64)
    if len(points) == 0:
        raise RuntimeError(f"No geolocation grid points found in {annotation_xml}")
    return points


# ============================================================================
# Download + extract one Sentinel-1 scene (cached: re-running skips re-download)
# ============================================================================

def download_and_extract_scene(product, session, download_root):
    scene_name = product.properties["sceneName"]
    scene_dir = download_root / scene_name

    cached = list(scene_dir.rglob("*.SAFE"))
    if cached:
        return cached[0]

    scene_dir.mkdir(parents=True, exist_ok=True)
    product.download(path=str(scene_dir), session=session)

    zips = list(scene_dir.glob("*.zip"))
    if not zips:
        raise RuntimeError(f"No ZIP downloaded for {scene_name}")

    extract_dir = scene_dir / "extracted"
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(extract_dir)

    safe_dirs = list(extract_dir.rglob("*.SAFE"))
    if not safe_dirs:
        raise RuntimeError(f"No .SAFE folder found after extracting {scene_name}")
    return safe_dirs[0]


def find_band_files(safe_dir, polarization):
    """Locate the measurement TIFF + annotation + calibration XML for one polarization ('vv' or 'vh')."""
    tag = f"-{polarization.lower()}-"
    tiff = next((f for f in (safe_dir / "measurement").glob("*.tif*") if tag in f.name.lower()), None)
    ann  = next((f for f in (safe_dir / "annotation").glob("*.xml") if tag in f.name.lower()), None)
    cal  = next((f for f in (safe_dir / "annotation" / "calibration").glob("calibration-*.xml") if tag in f.name.lower()), None)
    if not (tiff and ann and cal):
        raise RuntimeError(f"Missing {polarization.upper()} product files under {safe_dir}")
    return tiff, ann, cal

print("Download / extraction helpers ready.")

In [ ]:
# ============================================================================
# Radiometric calibration (DN -> sigma0 -> dB) + crop to the AOI.
#
# Sentinel-1 GRD calibration: sigma0 = DN^2 / LUT^2, where LUT is bilinearly
# interpolated from the calibration vectors (which are sparse in both pixel
# and line). This is the same maths v2 had hardcoded to one scene/band —
# here it takes the tiff/xml paths and AOI as arguments.
# ============================================================================

def calibrate_and_crop(tiff_path, annotation_xml, calibration_xml,
                        lon_min, lon_max, lat_min, lat_max, buffer_px=100):
    geo_points = read_geolocation_grid(annotation_xml)
    geo_xy = geo_points[:, 2:4]     # lon, lat
    geo_pixel, geo_line = geo_points[:, 0], geo_points[:, 1]

    aoi_corners = np.array([
        [lon_min, lat_min], [lon_max, lat_min],
        [lon_max, lat_max], [lon_min, lat_max],
    ], dtype=np.float64)

    corner_px = griddata(geo_xy, geo_pixel, aoi_corners, method="linear")
    corner_ln = griddata(geo_xy, geo_line, aoi_corners, method="linear")
    if np.any(np.isnan(corner_px)):
        fallback = griddata(geo_xy, geo_pixel, aoi_corners, method="nearest")
        corner_px[np.isnan(corner_px)] = fallback[np.isnan(corner_px)]
    if np.any(np.isnan(corner_ln)):
        fallback = griddata(geo_xy, geo_line, aoi_corners, method="nearest")
        corner_ln[np.isnan(corner_ln)] = fallback[np.isnan(corner_ln)]

    with rasterio.open(tiff_path) as src:
        img_w, img_h = src.width, src.height

    x_min = max(0, int(np.floor(corner_px.min())) - buffer_px)
    x_max = min(img_w, int(np.ceil(corner_px.max())) + buffer_px)
    y_min = max(0, int(np.floor(corner_ln.min())) - buffer_px)
    y_max = min(img_h, int(np.ceil(corner_ln.max())) + buffer_px)

    window = rasterio.windows.Window(col_off=x_min, row_off=y_min, width=x_max - x_min, height=y_max - y_min)
    with rasterio.open(tiff_path) as src:
        raw_crop = src.read(1, window=window)

    # ---- Calibration vectors: sparse grid of (line, pixel[], sigmaNought[]) ----
    cal_lines, cal_pixels, cal_sigma = [], [], []
    for elem in ET.parse(calibration_xml).getroot().iter():
        if local_tag(elem) != "calibrationVector":
            continue
        line_txt, pixel_txt, sigma_txt = child_text(elem, "line"), child_text(elem, "pixel"), child_text(elem, "sigmaNought")
        if None in (line_txt, pixel_txt, sigma_txt):
            continue
        cal_lines.append(int(line_txt))
        cal_pixels.append(np.fromstring(pixel_txt, sep=" ", dtype=np.float64))
        cal_sigma.append(np.fromstring(sigma_txt, sep=" ", dtype=np.float64))

    if not cal_lines:
        raise RuntimeError(f"No calibration vectors found in {calibration_xml}")

    order = np.argsort(cal_lines)
    cal_lines = np.array(cal_lines)[order]
    cal_pixels = [cal_pixels[i] for i in order]
    cal_sigma = [cal_sigma[i] for i in order]

    # Interpolate every calibration vector onto our actual crop columns.
    crop_columns = np.arange(x_min, x_max, dtype=np.float64)
    sigma_horizontal = np.array(
        [np.interp(crop_columns, px, sig) for px, sig in zip(cal_pixels, cal_sigma)],
        dtype=np.float32,
    )

    # Interpolate (in line/row direction) and apply: sigma0 = DN^2 / LUT^2
    raw_float = raw_crop.astype(np.float32)
    sigma0 = np.empty(raw_float.shape, dtype=np.float32)
    for local_row, global_row in enumerate(np.arange(y_min, y_max)):
        upper = np.searchsorted(cal_lines, global_row)
        if upper == 0:
            lut = sigma_horizontal[0]
        elif upper >= len(cal_lines):
            lut = sigma_horizontal[-1]
        else:
            lower = upper - 1
            l0, l1 = cal_lines[lower], cal_lines[upper]
            alpha = 0.0 if l1 == l0 else (global_row - l0) / (l1 - l0)
            lut = (1 - alpha) * sigma_horizontal[lower] + alpha * sigma_horizontal[upper]
        sigma0[local_row, :] = raw_float[local_row, :] ** 2 / (lut * lut + 1e-12)

    sigma0[raw_crop == 0] = np.nan  # no-data pixels

    sigma0_db = 10.0 * np.log10(np.maximum(sigma0, 1e-12))
    sigma0_db[~np.isfinite(sigma0)] = np.nan

    display_db = np.clip(sigma0_db, CFG.DB_MIN, CFG.DB_MAX)
    model_image = (display_db - CFG.DB_MIN) / (CFG.DB_MAX - CFG.DB_MIN) * 255.0
    model_image = np.nan_to_num(model_image, nan=0).astype(np.uint8)

    return {
        "model_image": model_image,
        "sigma0_db": sigma0_db,
        "x_min": x_min, "y_min": y_min, "x_max": x_max, "y_max": y_max,
        "geo_points": geo_points,
    }


def sample_sigma0(band_result, global_px, global_py):
    """sigma0 (dB) at an original-scene pixel/line coordinate, for a given calibrated band crop."""
    lx = int(round(global_px - band_result["x_min"]))
    ly = int(round(global_py - band_result["y_min"]))
    arr = band_result["sigma0_db"]
    if 0 <= ly < arr.shape[0] and 0 <= lx < arr.shape[1]:
        val = arr[ly, lx]
        return float(val) if np.isfinite(val) else np.nan
    return np.nan

print("Calibration helpers ready.")

In [ ]:
# ============================================================================
# Tiled inference: Sentinel-1 scenes are far larger than the 640x640 YOLO
# input, so we slide overlapping tiles across the scene and merge duplicate
# detections from the overlap with NMS.
# ============================================================================

def get_tile_positions(length, tile_size, stride):
    if length <= tile_size:
        return [0]
    positions = list(range(0, length - tile_size + 1, stride))
    if positions[-1] != length - tile_size:
        positions.append(length - tile_size)
    return positions


def numpy_nms(detections, iou_threshold=0.5):
    """[x1, y1, x2, y2, score] list -> same, duplicates suppressed. Pure numpy,
    so it works regardless of whether torchvision's compiled NMS op is available."""
    if len(detections) == 0:
        return []
    dets = np.array(detections, dtype=np.float32)
    x1, y1, x2, y2, scores = dets[:, 0], dets[:, 1], dets[:, 2], dets[:, 3], dets[:, 4]
    areas = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break
        xx1, yy1 = np.maximum(x1[i], x1[order[1:]]), np.maximum(y1[i], y1[order[1:]])
        xx2, yy2 = np.minimum(x2[i], x2[order[1:]]), np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0, xx2 - xx1) * np.maximum(0, yy2 - yy1)
        union = areas[i] + areas[order[1:]] - inter + 1e-8
        order = order[np.where(inter / union <= iou_threshold)[0] + 1]
    return dets[keep].tolist()


def run_tiled_inference(model, image, conf=CFG.CONF_THRESHOLD, tile_size=CFG.TILE_SIZE,
                         stride=CFG.TILE_STRIDE, iou=CFG.NMS_IOU):
    H, W = image.shape
    detections = []
    for y in get_tile_positions(H, tile_size, stride):
        for x in get_tile_positions(W, tile_size, stride):
            tile_bgr = cv2.cvtColor(image[y:y + tile_size, x:x + tile_size], cv2.COLOR_GRAY2BGR)
            result = model.predict(source=tile_bgr, imgsz=tile_size, conf=conf, iou=iou,
                                    device=0, verbose=False, max_det=1000)[0]
            if result.boxes is None or len(result.boxes) == 0:
                continue
            boxes = result.boxes.xyxy.cpu().numpy()
            confidences = result.boxes.conf.cpu().numpy()
            for (bx1, by1, bx2, by2), score in zip(boxes, confidences):
                detections.append([bx1 + x, by1 + y, bx2 + x, by2 + y, float(score)])
    return numpy_nms(detections, iou_threshold=iou)


# ============================================================================
# Geolocation: crop-pixel -> lon/lat, using the scene's geolocation tie points.
# ============================================================================

def build_pixel_to_lonlat(geo_points):
    tie_xy = geo_points[:, 0:2]   # pixel, line
    tie_lon, tie_lat = geo_points[:, 2], geo_points[:, 3]

    def pixel_to_lonlat(pixel, line):
        point = np.array([[pixel, line]], dtype=np.float64)
        lon = griddata(tie_xy, tie_lon, point, method="linear")[0]
        lat = griddata(tie_xy, tie_lat, point, method="linear")[0]
        if np.isnan(lon):
            lon = griddata(tie_xy, tie_lon, point, method="nearest")[0]
        if np.isnan(lat):
            lat = griddata(tie_xy, tie_lat, point, method="nearest")[0]
        return float(lon), float(lat)

    return pixel_to_lonlat


def detections_to_geo(detections, pixel_to_lonlat, x_min, y_min, model_name, scene_id):
    records = []
    for i, (x1, y1, x2, y2, conf) in enumerate(detections):
        cx_crop, cy_crop = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        global_px, global_py = cx_crop + x_min, cy_crop + y_min
        lon, lat = pixel_to_lonlat(global_px, global_py)
        records.append({
            "scene_id": scene_id, "model": model_name, "detection_id": f"{scene_id}_{model_name}_{i:04d}",
            "confidence": float(conf),
            "x1_crop": float(x1), "y1_crop": float(y1), "x2_crop": float(x2), "y2_crop": float(y2),
            "width_px": float(x2 - x1), "height_px": float(y2 - y1),
            "center_x_crop": cx_crop, "center_y_crop": cy_crop,
            "global_px": global_px, "global_py": global_py,
            "longitude": lon, "latitude": lat,
        })
    return pd.DataFrame(records)


# ============================================================================
# AIS matching: haversine distance from each detection to the nearest AIS cell.
# ============================================================================

def haversine_matrix(lat1, lon1, lat2, lon2):
    R = 6371000.0
    lat1r, lon1r = np.radians(np.asarray(lat1))[:, None], np.radians(np.asarray(lon1))[:, None]
    lat2r, lon2r = np.radians(np.asarray(lat2))[None, :], np.radians(np.asarray(lon2))[None, :]
    dlat, dlon = lat2r - lat1r, lon2r - lon1r
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def match_detections_to_ais(detection_df, ais_cells, radii=CFG.AIS_MATCH_RADII_M):
    out = detection_df.copy()
    if len(out) == 0 or len(ais_cells) == 0:
        out["nearest_ais_distance_m"] = np.nan
        return out, pd.DataFrame([{"Radius_m": r, "Detections": len(out), "Near_AIS": 0, "Near_AIS_pct": np.nan} for r in radii])

    distances = haversine_matrix(out["latitude"].values, out["longitude"].values, ais_cells["lat"].values, ais_cells["lon"].values)
    out["nearest_ais_distance_m"] = distances.min(axis=1)

    rows = []
    for r in radii:
        near = int((out["nearest_ais_distance_m"] <= r).sum())
        rows.append({"Radius_m": r, "Detections": len(out), "Near_AIS": near, "Near_AIS_pct": 100 * near / len(out)})
    return out, pd.DataFrame(rows)

print("Tiling / geolocation / AIS-matching helpers ready.")

In [ ]:
import requests
from google.colab import userdata

# ---- GFW token (Colab -> Secrets; must be enabled for this notebook) ------
GFW_TOKEN = userdata.get("fish")
if GFW_TOKEN is None:
    raise RuntimeError("GFW_TOKEN not found. Add it in Colab -> Secrets (key 'fish') and enable notebook access.")
GFW_TOKEN = GFW_TOKEN.strip()
print("GFW token loaded, looks like JWT:", GFW_TOKEN.startswith("eyJ"))


def fetch_ais_presence_for_scene(acquisition_dt, aoi_geojson, token, time_tolerance_hours=CFG.AIS_TIME_TOLERANCE_HOURS):
    """Query GFW AIS presence for the UTC day of `acquisition_dt`, then keep only the
    hourly bins within `time_tolerance_hours` of the actual SAR acquisition time.

    This is the piece v2 was missing: it queried a whole day and treated every AIS
    record in it as "supporting" the scene, even records many hours away from the
    actual overpass. Returns (all_records_that_day, records_within_time_window).
    """
    acquisition_dt = pd.to_datetime(acquisition_dt, utc=True)
    day_start = acquisition_dt.strftime("%Y-%m-%d")
    day_end = (acquisition_dt + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

    params = [
        ("spatial-resolution", "HIGH"),
        ("temporal-resolution", "HOURLY"),
        ("spatial-aggregation", "false"),
        ("datasets[0]", CFG.GFW_AIS_DATASET),
        ("date-range", f"{day_start},{day_end}"),
        ("format", "JSON"),
        ("group-by", "VESSEL_ID"),
    ]
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

    response = requests.post(CFG.GFW_REPORT_URL, headers=headers, params=params, json={"geojson": aoi_geojson}, timeout=180)
    if response.status_code != 200:
        raise RuntimeError(f"AIS query failed ({response.status_code}): {response.text[:500]}")

    records = []
    for entry in response.json().get("entries", []):
        if not isinstance(entry, dict):
            continue
        for key, value in entry.items():
            if isinstance(value, list):
                for row in value:
                    if isinstance(row, dict):
                        r = dict(row)
                        r["source_key"] = key
                        records.append(r)

    ais_df = pd.DataFrame(records)
    if len(ais_df) == 0 or "date" not in ais_df.columns:
        return ais_df, pd.DataFrame(columns=list(ais_df.columns) + ["hours_from_acquisition"])

    ais_df["bin_time"] = pd.to_datetime(ais_df["date"], utc=True, errors="coerce")
    ais_df["hours_from_acquisition"] = (ais_df["bin_time"] - acquisition_dt).dt.total_seconds() / 3600.0

    within_window = ais_df[ais_df["hours_from_acquisition"].abs() <= time_tolerance_hours].copy()
    return ais_df, within_window

print("AIS fetch helper ready. Tolerance window: +/-", CFG.AIS_TIME_TOLERANCE_HOURS, "hours around SAR acquisition time.")

## Part 5 — Select multiple Sentinel-1 scenes to validate on

v2 validated on a single manually-picked scene. Per your request, this searches the AOI over the configured date range, prefers scenes that share the same flight direction + relative orbit (so they're geometrically comparable), and selects `CFG.N_SCENES_TO_RUN` scenes spread across time.

In [ ]:
import asf_search as asf

search_results = asf.geo_search(
    platform=asf.PLATFORM.SENTINEL1,
    intersectsWith=CFG.AOI_WKT,
    start=CFG.SEARCH_START,
    end=CFG.SEARCH_END,
    beamMode=CFG.BEAM_MODE,
    processingLevel=asf.PRODUCT_TYPE.GRD_HD,
    polarization=CFG.POLARIZATION,
    maxResults=250,
)
print("Scenes found in AOI/date range:", len(search_results))

candidate_rows = []
for product in search_results:
    p = product.properties
    candidate_rows.append({
        "scene_name": p.get("sceneName"),
        "platform": p.get("platform"),
        "start_time": p.get("startTime"),
        "flight_direction": p.get("flightDirection"),
        "path_number": p.get("pathNumber"),
        "polarization": p.get("polarization"),
    })
candidate_df = pd.DataFrame(candidate_rows)
candidate_df["start_time"] = pd.to_datetime(candidate_df["start_time"], utc=True)
candidate_df = candidate_df.sort_values("start_time").reset_index(drop=True)

print("\nScenes per (flight_direction, path_number):")
print(candidate_df.groupby(["flight_direction", "path_number"]).size().sort_values(ascending=False))
candidate_df.head(10)

In [ ]:
scene_lookup = {p.properties["sceneName"]: p for p in search_results}

pool = candidate_df
if CFG.PREFER_SAME_GEOMETRY and len(candidate_df) > 0:
    group_sizes = candidate_df.groupby(["flight_direction", "path_number"]).size()
    best_direction, best_path = group_sizes.idxmax()
    pool = candidate_df[(candidate_df["flight_direction"] == best_direction) & (candidate_df["path_number"] == best_path)]
    print(f"Preferring geometry: flight_direction={best_direction}, path_number={best_path} ({len(pool)} scenes available)")

# Spread the selection evenly across the available timeline rather than just
# taking the most recent N (so validation isn't biased to one short period).
pool = pool.sort_values("start_time").reset_index(drop=True)
n = min(CFG.N_SCENES_TO_RUN, len(pool))
if n == 0:
    raise RuntimeError("No candidate scenes available — widen CFG.SEARCH_START/END or drop PREFER_SAME_GEOMETRY.")
pick_idx = np.linspace(0, len(pool) - 1, n).round().astype(int)
selected_df = pool.iloc[pick_idx].drop_duplicates(subset="scene_name").reset_index(drop=True)

selected_scenes = [scene_lookup[name] for name in selected_df["scene_name"]]

print(f"\nSelected {len(selected_scenes)} scenes for the multi-scene run:")
print(selected_df[["scene_name", "start_time", "flight_direction", "path_number"]].to_string(index=False))

# ---- Earthdata login (once, reused for every download in Part 6) ----------
from getpass import getpass

earthdata_username = input("Earthdata username: ")
earthdata_password = getpass("Earthdata password: ")
session = asf.ASFSession().auth_with_creds(earthdata_username, earthdata_password)
print("\nASF authentication successful.")

## Part 6 — Run the pipeline across all selected scenes (pre-filter)

For each scene: download → calibrate VV (and VH, if present) → run E0 & E1 → geolocate → fetch AIS within the acquisition-time window → match. One scene failing (a bad download, a missing band) is logged and skipped rather than stopping the whole run — with 6+ scenes, that matters.

Each scene's detections are cached to `CFG.S1_ROOT/per_scene/*.csv`, so re-running this cell after a Colab disconnect does not require re-downloading scenes already processed.

In [ ]:
DOWNLOAD_ROOT = CFG.S1_ROOT / "downloads"
PER_SCENE_ROOT = CFG.S1_ROOT / "per_scene"
DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
PER_SCENE_ROOT.mkdir(parents=True, exist_ok=True)

all_detections, all_ais_summaries, scene_log = [], [], []

for product in selected_scenes:
    scene_name = product.properties["sceneName"]
    acquisition_dt = pd.to_datetime(product.properties["startTime"], utc=True)
    cache_path = PER_SCENE_ROOT / f"{scene_name}_detections.csv"

    print("=" * 90)
    print(scene_name, "|", acquisition_dt)

    if cache_path.exists():
        print("  Cached, loading from disk.")
        scene_df = pd.read_csv(cache_path)
        all_detections.append(scene_df)
        scene_log.append({"scene_id": scene_name, "acquisition": acquisition_dt, "status": "ok (cached)"})
        continue

    try:
        safe_dir = download_and_extract_scene(product, session, DOWNLOAD_ROOT)

        vv_tiff, vv_ann, vv_cal = find_band_files(safe_dir, "vv")
        vv = calibrate_and_crop(vv_tiff, vv_ann, vv_cal, CFG.LON_MIN, CFG.LON_MAX, CFG.LAT_MIN, CFG.LAT_MAX, CFG.CROP_BUFFER_PX)

        vh = None
        try:
            vh_tiff, vh_ann, vh_cal = find_band_files(safe_dir, "vh")
            vh = calibrate_and_crop(vh_tiff, vh_ann, vh_cal, CFG.LON_MIN, CFG.LON_MAX, CFG.LAT_MIN, CFG.LAT_MAX, CFG.CROP_BUFFER_PX)
        except Exception as e:
            print("  VH unavailable:", e)

        pixel_to_lonlat = build_pixel_to_lonlat(vv["geo_points"])

        ais_raw, ais_window = fetch_ais_presence_for_scene(acquisition_dt, CFG.AOI_GEOJSON, GFW_TOKEN)
        ais_cells = (
            ais_window[["lat", "lon"]].dropna().astype(float).drop_duplicates()
            if len(ais_window) else pd.DataFrame(columns=["lat", "lon"])
        )
        print(f"  AIS records that day: {len(ais_raw)} | within +/-{CFG.AIS_TIME_TOLERANCE_HOURS}h of acquisition: "
              f"{len(ais_window)} | unique cells used: {len(ais_cells)}")

        scene_frames = []
        for model_name, model in [("E0", model_e0), ("E1", model_e1)]:
            dets = run_tiled_inference(model, vv["model_image"])
            geo = detections_to_geo(dets, pixel_to_lonlat, vv["x_min"], vv["y_min"], model_name, scene_name)
            geo, ais_summary = match_detections_to_ais(geo, ais_cells)
            ais_summary["scene_id"], ais_summary["model"] = scene_name, model_name
            all_ais_summaries.append(ais_summary)

            if vh is not None and len(geo):
                geo["vv_sigma0_db"] = [sample_sigma0(vv, px, py) for px, py in zip(geo["global_px"], geo["global_py"])]
                geo["vh_sigma0_db"] = [sample_sigma0(vh, px, py) for px, py in zip(geo["global_px"], geo["global_py"])]
                geo["vv_vh_ratio_db"] = geo["vv_sigma0_db"] - geo["vh_sigma0_db"]

            scene_frames.append(geo)
            print(f"  {model_name}: {len(dets)} detections")

        scene_df = pd.concat(scene_frames, ignore_index=True) if scene_frames else pd.DataFrame()
        scene_df.to_csv(cache_path, index=False)
        all_detections.append(scene_df)
        scene_log.append({"scene_id": scene_name, "acquisition": acquisition_dt, "status": "ok"})

    except Exception as e:
        print("  SCENE FAILED:", repr(e))
        scene_log.append({"scene_id": scene_name, "acquisition": acquisition_dt, "status": f"failed: {e}"})
        continue

master_detections = pd.concat(all_detections, ignore_index=True) if all_detections else pd.DataFrame()
master_ais_summary = pd.concat(all_ais_summaries, ignore_index=True) if all_ais_summaries else pd.DataFrame()
scene_log_df = pd.DataFrame(scene_log)

master_detections.to_csv(CFG.RESULTS_DIR / "master_detections_prefilter.csv", index=False)
scene_log_df.to_csv(CFG.RESULTS_DIR / "scene_processing_log.csv", index=False)

print("\n" + "=" * 90)
ok_count = (scene_log_df["status"].str.startswith("ok")).sum() if len(scene_log_df) else 0
print(f"Scenes processed OK: {ok_count} / {len(scene_log_df)}")
print(f"Total detections (pre-filter, both models, all scenes): {len(master_detections)}")

In [ ]:
# ---- Pre-filter summary, per model, at the main AIS radius (mirrors the
# "~2,048 detections / ~1,009 AIS-supported" style number from v2, now
# aggregated automatically across every scene instead of by hand on one) ----

prefilter_summary = (
    master_detections
    .assign(near_ais=lambda d: d["nearest_ais_distance_m"] <= CFG.AIS_MAIN_RADIUS_M)
    .groupby("model")
    .agg(scenes=("scene_id", "nunique"), total_detections=("detection_id", "count"), ais_supported=("near_ais", "sum"))
    .assign(ais_supported_pct=lambda d: 100 * d["ais_supported"] / d["total_detections"])
    .reset_index()
)
prefilter_summary.to_csv(CFG.RESULTS_DIR / "prefilter_summary_by_model.csv", index=False)

print(f"PRE-FILTER SUMMARY across {master_detections['scene_id'].nunique()} scenes (AIS match radius = {CFG.AIS_MAIN_RADIUS_M} m)")
prefilter_summary

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/SAR_Ship_Project")

print("Project exists:", root.exists())

for p in root.rglob("*.csv"):
    print(p)